# Day2: Build two core tables

目标：把 Sephora 原始数据整理成两张核心表：
1) product_facts（产品事实表）：一行一个 product_id
2) reviews（评论表）：一行一条评论

## 输出文件
- ../data_processed/product_facts.csv
- ../data_processed/reviews.csv

## 字段定义（最终以实际列为准）
### product_facts.csv
- product_id: 商品唯一ID
- product_name: 商品名
- brand_name: 品牌名
- category: 类目（如果存在）
- price: 价格（如果存在）
- ingredients / description / claims: 商品信息（如果存在）
- rating_avg: 平均评分（来自 products 或 reviews 聚合）
- review_cnt: 评论数（来自 reviews 聚合）

### reviews.csv
- product_id: 商品ID（用于关联 products）
- rating: 用户评分（若存在）
- review_text: 评论文本（若存在）
- review_date: 评论时间（若存在）

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data_raw")
OUT = Path("../data_processed")
OUT.mkdir(parents=True, exist_ok=True)

products = pd.read_csv(RAW / "product_info.csv")
print("products shape:", products.shape)
print(products.columns.tolist())
products.head(2)

products shape: (8494, 27)
['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,...,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0


In [2]:
review_files = sorted(RAW.glob("reviews_*.csv"))
review_files

[PosixPath('../data_raw/reviews_0-250.csv'),
 PosixPath('../data_raw/reviews_1250-end.csv'),
 PosixPath('../data_raw/reviews_250-500.csv'),
 PosixPath('../data_raw/reviews_500-750.csv'),
 PosixPath('../data_raw/reviews_750-1250.csv')]

In [3]:
dfs = []
for f in review_files:
    df = pd.read_csv(f)
    df["source_file"] = f.name
    dfs.append(df)

reviews_raw = pd.concat(dfs, ignore_index=True)
print("reviews_raw shape:", reviews_raw.shape)
print(reviews_raw.columns.tolist())
reviews_raw.head(2)

/var/folders/6_/_9mh8sxn79q7fm6bm97t_b5m0000gn/T/ipykernel_86187/2019221911.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/var/folders/6_/_9mh8sxn79q7fm6bm97t_b5m0000gn/T/ipykernel_86187/2019221911.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


reviews_raw shape: (1094411, 20)
['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd', 'source_file']


/var/folders/6_/_9mh8sxn79q7fm6bm97t_b5m0000gn/T/ipykernel_86187/2019221911.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


,Unnamed: 0,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,source_file
0,0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0,reviews_0-250.csv
1,1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,reviews_0-250.csv


In [4]:
def pick_col(cols, candidates):
    cols_lower = {c.lower(): c for c in cols}
    for cand in candidates:
        for c in cols:
            if cand in c.lower():
                return c
    return None

cols = reviews_raw.columns.tolist()

col_product_id = pick_col(cols, ["product_id", "productid"])
col_rating     = pick_col(cols, ["rating", "star", "score"])
col_text       = pick_col(cols, ["review_text", "review", "text", "content"])
col_date       = pick_col(cols, ["date", "time", "created", "submission"])

col_product_id, col_rating, col_text, col_date

('product_id', 'rating', 'review_text', 'submission_time')

In [5]:
reviews = reviews_raw.rename(columns={
    col_product_id: "product_id",
    col_rating: "rating",
    col_text: "review_text",
    col_date: "review_date",
})[["product_id", "rating", "review_text", "review_date"]].copy()

# 基础清洗：去空、去重（保守版）
reviews["review_text"] = reviews["review_text"].astype(str)
reviews = reviews.dropna(subset=["product_id"])
reviews = reviews.drop_duplicates()

print("reviews cleaned shape:", reviews.shape)
reviews.head(2)

reviews cleaned shape: (1093667, 4)


,product_id,rating,review_text,review_date
0,P504322,5,I use this with the Nudestix “Citrus Clean Bal...,2023-02-01
1,P420652,1,I bought this lip mask after reading the revie...,2023-03-21


In [6]:
agg = reviews.groupby("product_id").agg(
    review_cnt=("review_text", "size"),
    rating_avg=("rating", "mean")
).reset_index()

agg.head()

,product_id,review_cnt,rating_avg
0,P107306,253,4.031621
1,P114902,1529,4.419882
2,P12045,1686,4.443654
3,P122651,199,4.517588
4,P122661,798,4.525063


In [7]:
pcols = products.columns.tolist()
pcols

['product_id',
 'product_name',
 'brand_id',
 'brand_name',
 'loves_count',
 'rating',
 'reviews',
 'size',
 'variation_type',
 'variation_value',
 'variation_desc',
 'ingredients',
 'price_usd',
 'value_price_usd',
 'sale_price_usd',
 'limited_edition',
 'new',
 'online_only',
 'out_of_stock',
 'sephora_exclusive',
 'highlights',
 'primary_category',
 'secondary_category',
 'tertiary_category',
 'child_count',
 'child_max_price',
 'child_min_price']

In [10]:
# 先做最小可用版：product_id / product_name / brand_name + reviews聚合
base_cols = []
for c in ["product_id", "product_name", "brand_name"]:
    if c in products.columns:
        base_cols.append(c)

product_facts = products[base_cols].copy()
product_facts = product_facts.merge(agg, on="product_id", how="left")
# Final: product_facts（字段更完整，符合 Day2 要求）
product_facts_full = products[[
    "product_id",
    "product_name",
    "brand_name",
    "primary_category",
    "secondary_category",
    "tertiary_category",
    "price_usd",
    "sale_price_usd",
    "ingredients",
    "highlights",
    "rating"          # products 自带平均评分
]].copy()

# 合并 reviews 聚合结果（review_cnt / rating_avg）
product_facts_full = product_facts_full.merge(agg, on="product_id", how="left")

# review_cnt 缺失补 0
product_facts_full["review_cnt"] = product_facts_full["review_cnt"].fillna(0).astype(int)

# rating_avg：优先用 reviews 算的；没有就用 products.rating
product_facts_full["rating_avg"] = product_facts_full["rating_avg"].fillna(product_facts_full["rating"])

# 价格兜底：sale 为空用 price
product_facts_full["sale_price_usd"] = product_facts_full["sale_price_usd"].fillna(product_facts_full["price_usd"])

product_facts_full.head(2)

,product_id,product_name,brand_name,primary_category,secondary_category,tertiary_category,price_usd,sale_price_usd,ingredients,highlights,rating,review_cnt,rating_avg
0,P473671,Fragrance Discovery Set,19-69,Fragrance,Value & Gift Sets,Perfume Gift Sets,35.0,35.0,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD A...","['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",3.6364,0,3.6364
1,P473668,La Habana Eau de Parfum,19-69,Fragrance,Women,Perfume,195.0,195.0,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...","['Unisex/ Genderless Scent', 'Layerable Scent'...",4.1538,0,4.1538


In [12]:
reviews.to_csv(OUT / "reviews.csv", index=False)
product_facts_full.to_csv(OUT / "product_facts.csv", index=False)

print("saved:", OUT / "reviews.csv")
print("saved:", OUT / "product_facts.csv")

saved: ../data_processed/reviews.csv
saved: ../data_processed/product_facts.csv


In [13]:
import pandas as pd
from pathlib import Path

OUT = Path("../data_processed")

r = pd.read_csv(OUT / "reviews.csv")
p = pd.read_csv(OUT / "product_facts.csv")

print("reviews:", r.shape, r.columns.tolist())
print("product_facts:", p.shape, p.columns.tolist())
print("product_facts sample:\n", p.head(2))

reviews: (1093667, 4) ['product_id', 'rating', 'review_text', 'review_date']
product_facts: (8494, 13) ['product_id', 'product_name', 'brand_name', 'primary_category', 'secondary_category', 'tertiary_category', 'price_usd', 'sale_price_usd', 'ingredients', 'highlights', 'rating', 'review_cnt', 'rating_avg']
product_facts sample:
   product_id             product_name brand_name primary_category  \
0    P473671  Fragrance Discovery Set      19-69        Fragrance   
1    P473668  La Habana Eau de Parfum      19-69        Fragrance   

  secondary_category  tertiary_category  price_usd  sale_price_usd  \
0  Value & Gift Sets  Perfume Gift Sets       35.0            35.0   
1              Women            Perfume      195.0           195.0   

                                         ingredients  \
0  ['Capri Eau de Parfum:', 'Alcohol Denat. (SD A...   
1  ['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...   

                                          highlights  rating  review_cnt  \
0  [